# Experiments - Binary

Load necessary data for experiments

In [1]:
import pickle

# Load experiments
with open('experiments.pkl', 'rb') as f:
    experiments = pickle.load(f)

# Metrics Definition

Source: [scikit: make_scorer](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.make_scorer.html)

In [ ]:
from sklearn.metrics import make_scorer
from sklearn.metrics import recall_score, f1_score

# Classification Metrics
## binary
def recall_binary(y_true, y_pred):
    return recall_score(y_true, y_pred, pos_label=1)

def f1_binary(y_true, y_pred):
    return f1_score(y_true, y_pred, pos_label=1)

# ternary
def recall_macro(y_true, y_pred):
    return recall_score(y_true, y_pred, average='macro')

def f1_weighted(y_true, y_pred):
    return f1_score(y_true, y_pred, average='weighted')

# Custom metric functions for GridSearchCV to evaluate models

binary_scorers = {
    'recall': make_scorer(recall_binary),
    'f1': make_scorer(f1_binary)
}

ternary_scorers = {
    'recall_macro': make_scorer(recall_macro),
    'f1_weighted': make_scorer(f1_weighted)
}


## Define Experiments

Experiments for binary scenario:
1) Without preprocessing without data sampling
2) With preprocessing without data sampling
3) With preprocessing with under sampling
4) With preprocessing with hybrid sampling

In [3]:
from sklearn.model_selection import GridSearchCV

def grid_search_experiment(experiment, model, param_grid, experiment_type):
    """
    Perform GridSearchCV on the given experiment configuration using a specified model.
    
    Args:
        experiment: A dictionary containing 'sets', 'use_preprocessor', and 'sampling' keys.
        model: An instance of a scikit-learn estimator (e.g., RandomForestClassifier, MLPClassifier, SVC).
        param_grid: Dictionary of hyperparameters for GridSearchCV specific to the model.
        experiment_type: String indicating 'binary' or 'ternary' classification to select appropriate metrics.
    
    Returns:
        grid_search: Fitted GridSearchCV object with the best parameters and results.
    """
    # Extract datasets from the experiment
    X_train = experiment['sets']['X_train']
    y_train = experiment['sets']['y_train']
    X_val = experiment['sets']['X_val']
    y_val = experiment['sets']['y_val']

    # Select appropriate scorers and refit metric based on experiment type
    if experiment_type == 'binary':
        scorers = binary_scorers
        refit_metric = 'recall'
    else:  # experiment_type == 'ternary'
        scorers = ternary_scorers
        refit_metric = 'recall_macro'

    # Setup GridSearchCV with the provided model and custom metrics
    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=5,
        n_jobs=-1,
        scoring=scorers,
        refit=refit_metric,  # Optimize based on recall for binary or recall_macro for ternary
        verbose=1,
        return_train_score=True
    )

    # Fit GridSearchCV on training data
    grid_search.fit(X_train, y_train)

    # Print the best parameters and score for the refit metric
    print(f"Best parameters (based on {refit_metric}): {grid_search.best_params_}")
    print(f"Best cross-validation {refit_metric} score: {grid_search.best_score_:.4f}")
    
    # Access and print results for all metrics from cv_results_
    results = grid_search.cv_results_
    for scorer_name in scorers.keys():
        mean_score = results[f'mean_test_{scorer_name}'][grid_search.best_index_]
        std_score = results[f'std_test_{scorer_name}'][grid_search.best_index_]
        print(f"Best model mean test {scorer_name}: {mean_score:.4f} (+/- {std_score * 2:.4f})")

    # Evaluate on validation set for all metrics
    print("\nValidation set scores with best model:")
    for scorer_name, scorer in scorers.items():
        val_score = scorer(grid_search.best_estimator_, X_val, y_val)
        print(f"{scorer_name}: {val_score:.4f}")

    return grid_search

# Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

Parameter grid

In [ ]:
rf_param_grid = {
    'n_estimators': [50, 100, 200], 
    'max_depth': [None, 10, 20], 
    'min_samples_split': [5, 10],
    'min_samples_leaf': [2, 4]
}

1) Without preprocessing without data sampling

In [ ]:
# Initialize the model
rf_model = RandomForestClassifier(random_state=42)

# Perform grid search
rf_no_pre_no_sampling_grid_search = grid_search_experiment(
    experiments["binary"]["no_pre_no_sampling"],
    rf_model,
    rf_param_grid,
    "binary"
)

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best parameters (based on recall): {'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 50}
Best cross-validation recall score: 0.1546
Best model mean test recall: 0.1546 (+/- 0.0061)
Best model mean test f1: 0.2416 (+/- 0.0083)

Validation set scores with best model:
recall: 0.1624
f1: 0.2514


2) With preprocessing without data sampling

In [ ]:
# Initialize the model
rf_model = RandomForestClassifier(random_state=42)

# Perform grid search
rf_pre_no_sampling_grid_search = grid_search_experiment(
    experiments["binary"]["pre_no_sampling"],
    rf_model,
    rf_param_grid,
    "binary"
)

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best parameters (based on recall): {'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 50}
Best cross-validation recall score: 0.1528
Best model mean test recall: 0.1528 (+/- 0.0070)
Best model mean test f1: 0.2391 (+/- 0.0106)

Validation set scores with best model:
recall: 0.1638
f1: 0.2525


3) With preprocessing with under sampling

In [ ]:
# Initialize the model
rf_model = RandomForestClassifier(random_state=42)

# Perform grid search
rf_pre_undersampling_grid_search = grid_search_experiment(
    experiments["binary"]["pre_undersampling"],
    rf_model,
    rf_param_grid,
    "binary"
)

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best parameters (based on recall): {'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 50}
Best cross-validation recall score: 0.6386
Best model mean test recall: 0.6386 (+/- 0.0274)
Best model mean test f1: 0.6157 (+/- 0.0707)

Validation set scores with best model:
recall: 0.6311
f1: 0.3204


4) With preprocessing with hybrid sampling

In [24]:
# Initialize the model
rf_model = RandomForestClassifier(random_state=42)

# Perform grid search
rf_pre_hybrid_sampling_grid_search = grid_search_experiment(
    experiments["binary"]["pre_hybrid_sampling"],
    rf_model,
    rf_param_grid,
    "binary"
)

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best parameters (based on recall): {'max_depth': 20, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100}
Best cross-validation recall score: 0.9431
Best model mean test recall: 0.9431 (+/- 0.1869)
Best model mean test f1: 0.9394 (+/- 0.0966)

Validation set scores with best model:
recall: 0.6869
f1: 0.4622


# SVM

- Due to long processing time, a randomized grid search was used solely for the SVM model.
- The long duration was potentially because of the chosen kernel (specifically "rbf").
- A decision of changing both the grid_search algorithm as well as limiting the parameter grid in order to save some compute time. 
- The changes made resulted in compute time of 2 plus hours to 2 minutes, achieving good results.

Source: [sklearn: RandomizedSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html)

In [25]:
from sklearn.model_selection import RandomizedSearchCV

def Randomized_grid_search_experiment(experiment, model, param_grid, experiment_type):
    """
    Perform GridSearchCV on the given experiment configuration using a specified model.
    
    Args:
        experiment: A dictionary containing 'sets', 'use_preprocessor', and 'sampling' keys.
        model: An instance of a scikit-learn estimator (e.g., RandomForestClassifier, MLPClassifier, SVC).
        param_grid: Dictionary of hyperparameters for GridSearchCV specific to the model.
        experiment_type: String indicating 'binary' or 'ternary' classification to select appropriate metrics.
    
    Returns:
        grid_search: Fitted GridSearchCV object with the best parameters and results.
    """
    # Extract datasets from the experiment
    X_train = experiment['sets']['X_train']
    y_train = experiment['sets']['y_train']
    X_val = experiment['sets']['X_val']
    y_val = experiment['sets']['y_val']

    # Select appropriate scorers and refit metric based on experiment type
    if experiment_type == 'binary':
        scorers = binary_scorers
        refit_metric = 'recall'
    else:  # experiment_type == 'ternary'
        scorers = ternary_scorers
        refit_metric = 'recall_macro'

    """
    # Setup GridSearchCV with the provided model and custom metrics
    rand_grid_search = RandomizedSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=5,
        n_jobs=-1,
        scoring=scorers,
        refit=refit_metric,  # Optimize based on recall for binary or recall_macro for ternary
        verbose=1,
        return_train_score=True
    )
    """
    rand_grid_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_grid,  # Changed from param_grid to param_distributions for clarity
        n_iter=20,  # Sample 20 combinations
        cv=5,
        n_jobs=-1,
        scoring=scorers,
        refit=refit_metric,
        verbose=1,
        return_train_score=True
    )

    # Fit GridSearchCV on training data
    rand_grid_search.fit(X_train, y_train)

    # Print the best parameters and score for the refit metric
    print(f"Best parameters (based on {refit_metric}): {rand_grid_search.best_params_}")
    print(f"Best cross-validation {refit_metric} score: {rand_grid_search.best_score_:.4f}")
    
    # Access and print results for all metrics from cv_results_
    results = rand_grid_search.cv_results_
    for scorer_name in scorers.keys():
        mean_score = results[f'mean_test_{scorer_name}'][rand_grid_search.best_index_]
        std_score = results[f'std_test_{scorer_name}'][rand_grid_search.best_index_]
        print(f"Best model mean test {scorer_name}: {mean_score:.4f} (+/- {std_score * 2:.4f})")

    # Evaluate on validation set for all metrics
    print("\nValidation set scores with best model:")
    for scorer_name, scorer in scorers.items():
        val_score = scorer(rand_grid_search.best_estimator_, X_val, y_val)
        print(f"{scorer_name}: {val_score:.4f}")

    return rand_grid_search

In [ ]:
from sklearn.svm import SVC
from sklearn.svm import LinearSVC
from scipy.stats import uniform

Parameter grid

In [ ]:
svm_param_grid = {
    'C': uniform(0.01, 100),  # Continuous distribution for broader search # could've used [0.001, 0.01, 0.1, 1, 10, 100],
    'tol': [1e-4, 1e-3, 1e-2],  # Tolerance for convergence
    'max_iter': [1000, 5000, 10000],  # Iteration limits to ensure convergence
    'class_weight': ['balanced']  # Handle data imbalance
}

1) Without preprocessing without data sampling

In [ ]:
# Initialize the model
svm_model = LinearSVC()

# Perform randomized grid search 
svm_no_pre_no_sampling_rand_search = Randomized_grid_search_experiment(
    experiments["binary"]["no_pre_no_sampling"],
    svm_model,
    svm_param_grid,
    "binary"
)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best parameters (based on recall): {'C': 64.76464346912583, 'class_weight': 'balanced', 'max_iter': 1000, 'tol': 0.01}
Best cross-validation recall score: 0.7618
Best model mean test recall: 0.7618 (+/- 0.0085)
Best model mean test f1: 0.4496 (+/- 0.0044)

Validation set scores with best model:
recall: 0.7567
f1: 0.4452


2) With preprocessing without data sampling

In [29]:
"""
# Initialize the model
svm_model = SVC()

# Perform grid search
svm_pre_no_sampling_grid_search = grid_search_experiment(
    experiments["binary"]["pre_no_sampling"],
    svm_model,
    svm_param_grid,
    "binary"
)
"""


# Initialize the model
svm_model = LinearSVC()

# Perform randomized grid search 
svm_pre_no_sampling_rand_search = Randomized_grid_search_experiment(
    experiments["binary"]["pre_no_sampling"],
    svm_model,
    svm_param_grid,
    "binary"
)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best parameters (based on recall): {'C': 7.743518535575534, 'class_weight': 'balanced', 'max_iter': 1000, 'tol': 0.001}
Best cross-validation recall score: 0.7665
Best model mean test recall: 0.7665 (+/- 0.0105)
Best model mean test f1: 0.4497 (+/- 0.0053)

Validation set scores with best model:
recall: 0.7658
f1: 0.4464


3) With preprocessing with under sampling

In [ ]:
# Initialize the model
svm_model = LinearSVC()

# Perform randomized grid search 
svm_pre_undersampling_rand_search = Randomized_grid_search_experiment(
    experiments["binary"]["pre_undersampling"],
    svm_model,
    svm_param_grid,
    "binary"
)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best parameters (based on recall): {'C': 48.74495964452362, 'class_weight': 'balanced', 'max_iter': 5000, 'tol': 0.0001}
Best cross-validation recall score: 0.6567
Best model mean test recall: 0.6567 (+/- 0.0433)
Best model mean test f1: 0.6141 (+/- 0.0645)

Validation set scores with best model:
recall: 0.6490
f1: 0.3881


4) With preprocessing with hybrid sampling

In [ ]:
# Initialize the model
svm_model = LinearSVC()

# Perform randomized grid search 
svm_pre_hybrid_sampling_grid_search = Randomized_grid_search_experiment(
    experiments["binary"]["pre_hybrid_sampling"],
    svm_model,
    svm_param_grid,
    "binary"
)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best parameters (based on recall): {'C': 31.763401824210614, 'class_weight': 'balanced', 'max_iter': 1000, 'tol': 0.001}
Best cross-validation recall score: 0.8572
Best model mean test recall: 0.8572 (+/- 0.0061)
Best model mean test f1: 0.8670 (+/- 0.0034)

Validation set scores with best model:
recall: 0.8068
f1: 0.4383


# MLP

In [9]:
from sklearn.neural_network import MLPClassifier

In [16]:
nn_mlp_param_grid = {
    'hidden_layer_sizes': [(64, 32), (30, 15)],
    'activation': ['relu'],  # ['relu', 'tanh']
    'solver': ['adam'],  #
    'learning_rate': ['constant'], 
    'learning_rate_init': [0.001, 0.01], 
    'alpha': [0.0001, 0.01],  #  [0.0001, 0.01] # 0.02
    'max_iter': [1000],  
    'early_stopping': [True],  # Prevent overfit and save time
    'validation_fraction': [0.1],  # Fixed to 0.1
    'batch_size': ['auto'],  # [32, 'auto']
    'random_state': [42]  # Reproducibility
}

1) Without preprocessing without data sampling

In [11]:
# Initialize the model
mlp_model = MLPClassifier()

# Perform grid search
mlp_no_pre_no_sampling_grid_search = grid_search_experiment(
    experiments["binary"]["no_pre_no_sampling"],
    mlp_model,
    nn_mlp_param_grid,
    "binary"
)

Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best parameters (based on recall): {'activation': 'relu', 'alpha': 0.01, 'batch_size': 'auto', 'early_stopping': True, 'hidden_layer_sizes': (64, 32), 'learning_rate': 'constant', 'learning_rate_init': 0.001, 'max_iter': 1000, 'random_state': 42, 'solver': 'adam', 'validation_fraction': 0.1}
Best cross-validation recall score: 0.1654
Best model mean test recall: 0.1654 (+/- 0.0183)
Best model mean test f1: 0.2552 (+/- 0.0196)

Validation set scores with best model:
recall: 0.1772
f1: 0.2701


2) With preprocessing without data sampling

In [12]:
# Initialize the model
mlp_model = MLPClassifier()

# Perform grid search
mlp_pre_no_sampling_grid_search = grid_search_experiment(
    experiments["binary"]["pre_no_sampling"],
    mlp_model,
    nn_mlp_param_grid,
    "binary"
)

Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best parameters (based on recall): {'activation': 'relu', 'alpha': 0.01, 'batch_size': 'auto', 'early_stopping': True, 'hidden_layer_sizes': (30, 15), 'learning_rate': 'constant', 'learning_rate_init': 0.01, 'max_iter': 1000, 'random_state': 42, 'solver': 'adam', 'validation_fraction': 0.1}
Best cross-validation recall score: 0.1733
Best model mean test recall: 0.1733 (+/- 0.0468)
Best model mean test f1: 0.2640 (+/- 0.0538)

Validation set scores with best model:
recall: 0.1801
f1: 0.2733


3) With preprocessing with under sampling

In [15]:
# Initialize the model
mlp_model = MLPClassifier()

# Perform grid search
mlp_pre_undersampling_grid_search = grid_search_experiment(
    experiments["binary"]["pre_undersampling"],
    mlp_model,
    nn_mlp_param_grid,
    "binary"
)

Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best parameters (based on recall): {'activation': 'relu', 'alpha': 0.0001, 'batch_size': 'auto', 'early_stopping': True, 'hidden_layer_sizes': (64, 32), 'learning_rate': 'constant', 'learning_rate_init': 0.001, 'max_iter': 1000, 'random_state': 42, 'solver': 'adam', 'validation_fraction': 0.1}
Best cross-validation recall score: 0.5981
Best model mean test recall: 0.5981 (+/- 0.0573)
Best model mean test f1: 0.5856 (+/- 0.1104)

Validation set scores with best model:
recall: 0.5900
f1: 0.3101


4) With preprocessing with hybrid sampling

In [18]:
# Initialize the model
mlp_model = MLPClassifier()

# Perform grid search
mlp_pre_hybrid_sampling_grid_search = grid_search_experiment(
    experiments["binary"]["pre_hybrid_sampling"],
    mlp_model,
    nn_mlp_param_grid,
    "binary"
)

Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best parameters (based on recall): {'activation': 'relu', 'alpha': 0.0001, 'batch_size': 'auto', 'early_stopping': True, 'hidden_layer_sizes': (64, 32), 'learning_rate': 'constant', 'learning_rate_init': 0.001, 'max_iter': 1000, 'random_state': 42, 'solver': 'adam', 'validation_fraction': 0.1}
Best cross-validation recall score: 0.9187
Best model mean test recall: 0.9187 (+/- 0.0254)
Best model mean test f1: 0.8988 (+/- 0.0133)

Validation set scores with best model:
recall: 0.8037
f1: 0.4212


# Best Models - Test

- Select the best 2 models.
- Run Experiments with parameters that created the best scores.
- Remove random if that was done to ensure it can have standard deviation.

In [32]:
import numpy as np

In [ ]:
def evaluate_model_with_fixed_params(grid_search_obj, X_train, y_train, X_test, y_test, n_runs=30, experiment_type='binary'):
    """
    Train and evaluate the best model from a grid search object multiple times using bootstrapped training data,
    and test on a fixed test set. Calculate mean and std of metrics using provided scorers.
    
    Args:
        grid_search_obj: Fitted GridSearchCV or RandomizedSearchCV object containing the best_estimator_.
        X_train: Training set features to bootstrap from.
        y_train: Training set labels to bootstrap from.
        X_test: Fixed test set features.
        y_test: Fixed test set labels.
        n_runs: Number of times to train and evaluate the model.
        experiment_type: 'binary' or 'ternary' classification to select appropriate metrics.
    
    Returns:
        dict: Mean and std of metrics.
    """
    # Select appropriate scorers based on experiment type
    scorers = binary_scorers if experiment_type == 'binary' else ternary_scorers
    metric_names = list(scorers.keys())
    scores = {name: [] for name in metric_names}
    
    # Extract the best estimator (model with best parameters) from grid search object
    best_model = grid_search_obj.best_estimator_
    
    n_samples = len(X_train)
    
    for i in range(n_runs):
        # Bootstrap: Randomly sample with replacement from training data
        # Use random seed for reproducibility of each run
        np.random.seed(i)
        indices = np.random.choice(n_samples, size=n_samples, replace=True)
        X_train_bootstrap = X_train[indices]
        y_train_bootstrap = y_train[indices]
        
        # Train
        best_model.fit(X_train_bootstrap, y_train_bootstrap)
        
        # Calculate scores
        for metric_name, scorer in scorers.items():
            score = scorer(best_model, X_test, y_test)
            scores[metric_name].append(score)
    
    # Compute mean and std for each metric
    results = {}
    for metric_name in metric_names:
        mean_score = np.mean(scores[metric_name])
        std_score = np.std(scores[metric_name])
        results[f"{metric_name}_mean"] = mean_score
        results[f"{metric_name}_std"] = std_score
        print(f"{metric_name} - Mean: {mean_score:.4f}, Std: {std_score:.4f}")
    
    return results

Best Models in Binary - Recall:
- SVM: pre_hybrid_sampling (0.8068)
- MLP pre_hybrid_sampling (0.8037)

In [34]:
mlp_pre_hybrid_sampling_grid_search

GridSearchCV(cv=5, estimator=MLPClassifier(), n_jobs=-1,
             param_grid={'activation': ['relu'], 'alpha': [0.0001, 0.01],
                         'batch_size': ['auto'], 'early_stopping': [True],
                         'hidden_layer_sizes': [(64, 32), (30, 15)],
                         'learning_rate': ['constant'],
                         'learning_rate_init': [0.001, 0.01],
                         'max_iter': [1000], 'random_state': [42],
                         'solver': ['adam'], 'validation_fraction': [0.1]},
             refit='recall', return_train_score=True,
             scoring={'f1': make_scorer(f1_binary, response_method='predict'),
                      'recall': make_scorer(recall_binary, response_method='predict')},
             verbose=1)

In [35]:
svm_pre_hybrid_sampling_grid_search

RandomizedSearchCV(cv=5, estimator=LinearSVC(), n_iter=20, n_jobs=-1,
                   param_distributions={'C': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x000001D45294DD30>,
                                        'class_weight': ['balanced'],
                                        'max_iter': [1000, 5000, 10000],
                                        'tol': [0.0001, 0.001, 0.01]},
                   refit='recall', return_train_score=True,
                   scoring={'f1': make_scorer(f1_binary, response_method='predict'),
                            'recall': make_scorer(recall_binary, response_method='predict')},
                   verbose=1)

SVM

LinearSVC(C=31.763401824210614, class_weight='balanced', tol=0.001)


In [ ]:
# Run the evaluation over 30 fits with fixed parameters
results = evaluate_model_with_fixed_params(
    grid_search_obj=svm_pre_hybrid_sampling_grid_search,  # Pass the grid search object
    X_train = experiments["binary"]["pre_hybrid_sampling"]['sets']['X_train'],
    y_train = experiments["binary"]["pre_hybrid_sampling"]['sets']['y_train'],
    X_test  = experiments["binary"]["pre_hybrid_sampling"]['sets']['X_test'],
    y_test  = experiments["binary"]["pre_hybrid_sampling"]['sets']['y_test'],
    n_runs=30,
    experiment_type='binary'
)

recall - Mean: 0.7957, Std: 0.0009
f1 - Mean: 0.4359, Std: 0.0004


MLP

MLPClassifier(early_stopping=True, hidden_layer_sizes=(64, 32), max_iter=1000, random_state=42)

In [42]:
# Run the evaluation over 30 fits with fixed parameters
results = evaluate_model_with_fixed_params(
    grid_search_obj=mlp_pre_hybrid_sampling_grid_search,  # Pass the grid search object
    X_train = experiments["binary"]["pre_hybrid_sampling"]['sets']['X_train'],
    y_train = experiments["binary"]["pre_hybrid_sampling"]['sets']['y_train'],
    X_test  = experiments["binary"]["pre_hybrid_sampling"]['sets']['X_test'],
    y_test  = experiments["binary"]["pre_hybrid_sampling"]['sets']['y_test'],
    n_runs=30,
    experiment_type='binary'
)

recall - Mean: 0.7835, Std: 0.0103
f1 - Mean: 0.4214, Std: 0.0033
